# 🔧 Image Preprocessing Pipeline

This notebook demonstrates the preprocessing steps:
1. Image resizing to 256×256
2. Tensor conversion
3. Data augmentation effects
4. PyTorch Dataset and DataLoader setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from PIL import Image

from src.dataset import NormalDataset, TestDataset, get_transforms
from src.preprocessing import preprocess_image, tensor_to_image

DATA_DIR = '../data/mvtec_anomaly_detection/bottle'

## 1. Basic Preprocessing

In [ ]:
# Load a sample image
img_dir = os.path.join(DATA_DIR, 'train', 'good')
img_path = os.path.join(img_dir, sorted(os.listdir(img_dir))[0])
original = Image.open(img_path).convert('RGB')

print(f'Original size: {original.size}')

# Preprocess
tensor = preprocess_image(original, image_size=256)
print(f'Tensor shape: {tensor.shape}')
print(f'Value range: [{tensor.min():.4f}, {tensor.max():.4f}]')

# Convert back for visualization
reconstructed = tensor_to_image(tensor)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original)
axes[0].set_title(f'Original ({original.size[0]}×{original.size[1]})')
axes[0].axis('off')

axes[1].imshow(reconstructed)
axes[1].set_title(f'Preprocessed (256×256)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2. Data Augmentation Effects

In [ ]:
# Show augmentation effects
augment_transform = get_transforms(256, augment=True)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Data Augmentation Samples (same source image)', fontsize=14)

for ax in axes.flat:
    augmented = augment_transform(original)
    ax.imshow(tensor_to_image(augmented))
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Dataset and DataLoader

In [ ]:
# Create dataset
transform = get_transforms(256, augment=False)
dataset = NormalDataset(os.path.join(DATA_DIR, 'train', 'good'), transform=transform)
print(f'Dataset size: {len(dataset)}')

# Train/val split
val_size = int(0.15 * len(dataset))
train_size = len(dataset) - val_size
train_set, val_set = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
print(f'Train: {len(train_set)}, Val: {len(val_set)}')

# DataLoader
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
batch = next(iter(train_loader))
print(f'Batch shape: {batch.shape}')
print(f'Batch dtype: {batch.dtype}')
print(f'Value range: [{batch.min():.4f}, {batch.max():.4f}]')

## 4. Test Dataset with Labels and Masks

In [ ]:
# Test dataset
test_dataset = TestDataset(
    test_dir=os.path.join(DATA_DIR, 'test'),
    ground_truth_dir=os.path.join(DATA_DIR, 'ground_truth'),
    transform=transform,
)
print(f'Test dataset size: {len(test_dataset)}')

# Show samples
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(3):
    idx = i * (len(test_dataset) // 3)
    image, label, mask, path = test_dataset[idx]
    
    axes[i, 0].imshow(tensor_to_image(image))
    axes[i, 0].set_title(f'Image ({"Normal" if label == 0 else "Defective"})')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(mask[0], cmap='gray')
    axes[i, 1].set_title('Ground Truth Mask')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(tensor_to_image(image))
    axes[i, 2].imshow(mask[0], cmap='Reds', alpha=0.4)
    axes[i, 2].set_title('Overlay')
    axes[i, 2].axis('off')
    
    axes[i, 3].text(0.5, 0.5, f'Label: {label}\nPath: {os.path.basename(path)}',
                    ha='center', va='center', fontsize=11, transform=axes[i, 3].transAxes)
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()